# Домашнее задание 5: Временная стабилизация масок (Anti-Flicker)

**Вариант A**: Система временной стабилизации масок для кадр-по-кадр сегментации

## Цель
Освоить практические методы видео-сегментации, изучить способы стабилизации масок во времени и оценить влияние временных артефактов на итоговое качество сегментации.


## 1. Импорт библиотек и настройка окружения


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import torchvision
from torchvision.models.segmentation import deeplabv3_resnet50
from scipy.ndimage import median_filter
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Настройка matplotlib
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['figure.dpi'] = 100

# Проверка CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Используемое устройство: {device}')


## 2. Загрузка видео и извлечение кадров


In [ ]:
def load_video_frames(video_path, max_frames=None):
    """Загружает кадры из видеофайла."""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if max_frames:
        total_frames = min(total_frames, max_frames)
    
    frames = []
    print(f'Загрузка {total_frames} кадров из видео...')
    
    for i in tqdm(range(total_frames)):
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)
    
    cap.release()
    print(f'Загружено {len(frames)} кадров, FPS: {fps}')
    return frames, fps

# Загружаем видео
video_path = Path('../HW1/tall.mp4')
frames, fps = load_video_frames(video_path, max_frames=100)

# Показываем первый кадр
plt.figure(figsize=(10, 6))
plt.imshow(frames[0])
plt.title('Первый кадр видео')
plt.axis('off')
plt.savefig('results/first_frame.png', bbox_inches='tight', dpi=150)
plt.show()


## 3. Загрузка предобученной модели сегментации

Используем предобученную модель DeepLabV3 с ResNet-50 для семантической сегментации.


In [ ]:
def load_segmentation_model():
    """Загружает предобученную модель DeepLabV3."""
    model = deeplabv3_resnet50(pretrained=True)
    model = model.to(device)
    model.eval()
    return model

def preprocess_frame(frame):
    """Предобработка кадра для модели сегментации."""
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    return transform(frame).unsqueeze(0).to(device)

# Загружаем модель
print('Загрузка модели сегментации...')
model = load_segmentation_model()
print('Модель загружена успешно!')


## 4. Генерация масок для всех кадров

Прогоняем каждый кадр через модель и получаем вероятностные маски для класса "человек" (class 15 в COCO).


In [ ]:
def generate_masks(model, frames, target_class=15):
    """Генерирует маски для всех кадров. target_class=15 это человек в COCO."""
    masks = []
    print(f'Генерация масок для {len(frames)} кадров...')
    
    with torch.no_grad():
        for frame in tqdm(frames):
            input_tensor = preprocess_frame(frame)
            output = model(input_tensor)['out']
            probs = torch.softmax(output, dim=1)[0]
            mask = probs[target_class].cpu().numpy()
            mask_resized = cv2.resize(mask, (frame.shape[1], frame.shape[0]))
            masks.append(mask_resized)
    
    return masks

# Генерируем маски
original_masks = generate_masks(model, frames)
print(f'Сгенерировано {len(original_masks)} масок')
print(f'Размер маски: {original_masks[0].shape}')


## 5. Визуализация исходных масок


In [ ]:
def visualize_masks(frames, masks, indices=[0, 25, 50, 75], title_prefix='Маска'):
    """Визуализирует маски на выбранных кадрах."""
    fig, axes = plt.subplots(2, len(indices), figsize=(20, 10))
    
    for i, idx in enumerate(indices):
        if idx >= len(frames):
            continue
        axes[0, i].imshow(frames[idx])
        axes[0, i].set_title(f'Кадр {idx}')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(masks[idx], cmap='hot', vmin=0, vmax=1)
        axes[1, i].set_title(f'{title_prefix} {idx}')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    return fig

# Визуализируем исходные маски
fig = visualize_masks(frames, original_masks, title_prefix='Исходная маска')
plt.savefig('results/original_masks.png', bbox_inches='tight', dpi=150)
plt.show()


## 6. Метрики временной нестабильности

Реализуем несколько метрик для оценки стабильности масок во времени:
1. **IoU между соседними кадрами** - классическая метрика согласованности
2. **Temporal Variance** - среднее абсолютное изменение маски
3. **Boundary Flickering** - изменение границ маски


In [ ]:
def compute_iou(mask1, mask2, threshold=0.5):
    """Вычисляет IoU между двумя масками."""
    m1 = (mask1 > threshold).astype(np.float32)
    m2 = (mask2 > threshold).astype(np.float32)
    
    intersection = np.sum(m1 * m2)
    union = np.sum(m1) + np.sum(m2) - intersection
    
    if union == 0:
        return 1.0
    return intersection / union

def compute_temporal_iou(masks, threshold=0.5):
    """Вычисляет IoU между последовательными масками."""
    ious = []
    for i in range(len(masks) - 1):
        iou = compute_iou(masks[i], masks[i+1], threshold)
        ious.append(iou)
    return np.array(ious)

def compute_temporal_variance(masks):
    """Вычисляет среднее абсолютное изменение маски по времени."""
    variances = []
    for i in range(len(masks) - 1):
        diff = np.abs(masks[i+1] - masks[i])
        variance = np.mean(diff)
        variances.append(variance)
    return np.array(variances)

def compute_boundary_flickering(masks, threshold=0.5):
    """Вычисляет изменение границ маски."""
    flickering = []
    for i in range(len(masks) - 1):
        m1 = (masks[i] > threshold).astype(np.uint8)
        m2 = (masks[i+1] > threshold).astype(np.uint8)
        
        contours1, _ = cv2.findContours(m1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours2, _ = cv2.findContours(m2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        edge1 = cv2.drawContours(np.zeros_like(m1), contours1, -1, 1, 2)
        edge2 = cv2.drawContours(np.zeros_like(m2), contours2, -1, 1, 2)
        
        diff = np.abs(edge1.astype(float) - edge2.astype(float))
        flickering.append(np.mean(diff))
    
    return np.array(flickering)

# Вычисляем метрики для исходных масок
print('Вычисление метрик нестабильности...')
original_ious = compute_temporal_iou(original_masks)
original_variance = compute_temporal_variance(original_masks)
original_flickering = compute_boundary_flickering(original_masks)

print(f'\\nМетрики исходных масок:')
print(f'Средний IoU: {np.mean(original_ious):.4f} ± {np.std(original_ious):.4f}')
print(f'Дисперсия: {np.mean(original_variance):.6f} ± {np.std(original_variance):.6f}')
print(f'Мерцание границ: {np.mean(original_flickering):.6f} ± {np.std(original_flickering):.6f}')


In [ ]:
def temporal_averaging(masks, window_size=5):
    """Усреднение масок по скользящему окну."""
    smoothed_masks = []
    half_window = window_size // 2
    
    for i in range(len(masks)):
        start = max(0, i - half_window)
        end = min(len(masks), i + half_window + 1)
        window_masks = masks[start:end]
        smoothed_mask = np.mean(window_masks, axis=0)
        smoothed_masks.append(smoothed_mask)
    
    return smoothed_masks

def temporal_median_filtering(masks, window_size=5):
    """Медианная фильтрация по времени."""
    masks_array = np.array(masks)
    smoothed_masks = []
    half_window = window_size // 2
    
    for i in range(len(masks)):
        start = max(0, i - half_window)
        end = min(len(masks), i + half_window + 1)
        window_masks = masks_array[start:end]
        smoothed_mask = np.median(window_masks, axis=0)
        smoothed_masks.append(smoothed_mask)
    
    return smoothed_masks

def exponential_smoothing(masks, alpha=0.3):
    """Экспоненциальное сглаживание. alpha контролирует силу сглаживания."""
    smoothed_masks = [masks[0].copy()]
    
    for i in range(1, len(masks)):
        smoothed_mask = alpha * masks[i] + (1 - alpha) * smoothed_masks[-1]
        smoothed_masks.append(smoothed_mask)
    
    return smoothed_masks

# Применяем все методы
print('Применение методов сглаживания...')
averaged_masks = temporal_averaging(original_masks, window_size=5)
print('Усреднение завершено')
median_masks = temporal_median_filtering(original_masks, window_size=5)
print('Медианная фильтрация завершена')
exponential_masks = exponential_smoothing(original_masks, alpha=0.3)
print('Экспоненциальное сглаживание завершено')


## 8. Сравнение результатов сглаживания


In [ ]:
def evaluate_smoothing_method(masks, method_name):
    """Оценивает качество метода сглаживания."""
    ious = compute_temporal_iou(masks)
    variance = compute_temporal_variance(masks)
    flickering = compute_boundary_flickering(masks)
    
    results = {
        'method': method_name,
        'mean_iou': float(np.mean(ious)),
        'std_iou': float(np.std(ious)),
        'mean_variance': float(np.mean(variance)),
        'std_variance': float(np.std(variance)),
        'mean_flickering': float(np.mean(flickering)),
        'std_flickering': float(np.std(flickering))
    }
    
    return results, ious, variance, flickering

# Оцениваем все методы
print('Оценка результатов сглаживания...\\n')

results_original, ious_orig, var_orig, flick_orig = evaluate_smoothing_method(
    original_masks, 'Original'
)
results_averaged, ious_avg, var_avg, flick_avg = evaluate_smoothing_method(
    averaged_masks, 'Temporal Averaging'
)
results_median, ious_med, var_med, flick_med = evaluate_smoothing_method(
    median_masks, 'Median Filtering'
)
results_exponential, ious_exp, var_exp, flick_exp = evaluate_smoothing_method(
    exponential_masks, 'Exponential Smoothing'
)

all_results = [results_original, results_averaged, results_median, results_exponential]

# Сохраняем результаты
with open('results/metrics.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Выводим таблицу
print('='*80)
print('РЕЗУЛЬТАТЫ СРАВНЕНИЯ МЕТОДОВ')
print('='*80)

for r in all_results:
    print(f"\\n{r['method']}:")
    print(f"  IoU:      {r['mean_iou']:.4f} ± {r['std_iou']:.4f}")
    print(f"  Variance: {r['mean_variance']:.6f} ± {r['std_variance']:.6f}")
    print(f"  Flicker:  {r['mean_flickering']:.6f} ± {r['std_flickering']:.6f}")

print('='*80)


## 9. Визуализация метрик


In [ ]:
# График метрик
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# IoU
axes[0].plot(ious_orig, label='Original', linewidth=2, alpha=0.7)
axes[0].plot(ious_avg, label='Temporal Averaging', linewidth=2, alpha=0.7)
axes[0].plot(ious_med, label='Median Filtering', linewidth=2, alpha=0.7)
axes[0].plot(ious_exp, label='Exponential Smoothing', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Номер кадра')
axes[0].set_ylabel('IoU')
axes[0].set_title('IoU между соседними кадрами')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Variance
axes[1].plot(var_orig, label='Original', linewidth=2, alpha=0.7)
axes[1].plot(var_avg, label='Temporal Averaging', linewidth=2, alpha=0.7)
axes[1].plot(var_med, label='Median Filtering', linewidth=2, alpha=0.7)
axes[1].plot(var_exp, label='Exponential Smoothing', linewidth=2, alpha=0.7)
axes[1].set_xlabel('Номер кадра')
axes[1].set_ylabel('Дисперсия')
axes[1].set_title('Временная дисперсия маски')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Flickering
axes[2].plot(flick_orig, label='Original', linewidth=2, alpha=0.7)
axes[2].plot(flick_avg, label='Temporal Averaging', linewidth=2, alpha=0.7)
axes[2].plot(flick_med, label='Median Filtering', linewidth=2, alpha=0.7)
axes[2].plot(flick_exp, label='Exponential Smoothing', linewidth=2, alpha=0.7)
axes[2].set_xlabel('Номер кадра')
axes[2].set_ylabel('Flickering')
axes[2].set_title('Мерцание границ маски')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/metrics_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


## 10. Визуальное сравнение до/после сглаживания


In [ ]:
def create_comparison(frames, orig_masks, smooth_masks, name, indices=[10, 30, 50, 70]):
    """Создает сравнительную визуализацию."""
    fig, axes = plt.subplots(3, len(indices), figsize=(20, 12))
    
    for i, idx in enumerate(indices):
        if idx >= len(frames):
            continue
        
        axes[0, i].imshow(frames[idx])
        axes[0, i].set_title(f'Кадр {idx}')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(orig_masks[idx], cmap='hot', vmin=0, vmax=1)
        axes[1, i].set_title('До сглаживания')
        axes[1, i].axis('off')
        
        axes[2, i].imshow(smooth_masks[idx], cmap='hot', vmin=0, vmax=1)
        axes[2, i].set_title(f'После ({name})')
        axes[2, i].axis('off')
    
    plt.tight_layout()
    return fig

# Визуализация лучшего метода
fig = create_comparison(frames, original_masks, exponential_masks, 'Exponential')
plt.savefig('results/comparison_best.png', bbox_inches='tight', dpi=150)
plt.show()


## 11. Инженерный вывод и анализ


In [ ]:
print('\\n' + '='*80)
print('ИНЖЕНЕРНЫЙ ВЫВОД')
print('='*80)

print('''
1. ПРОБЛЕМА МЕРЦАНИЯ МАСОК:
   - Покадровая сегментация создает нестабильные маски
   - Границы объектов "дрожат" даже при плавном движении
   - Особенно заметно на сложных текстурах

2. ЭФФЕКТИВНОСТЬ МЕТОДОВ:
   
   a) Temporal Averaging:
      ✓ Простой и быстрый
      ✓ Уменьшает высокочастотный шум
      ✗ Размывает границы при быстром движении
   
   b) Median Filtering:
      ✓ Справляется с выбросами
      ✓ Сохраняет резкие границы
      ✗ Вычислительно затратный
   
   c) Exponential Smoothing:
      ✓✓ ЛУЧШИЙ БАЛАНС
      ✓ Адаптивно реагирует на изменения
      ✓ Минимальная задержка
      ✓ Один параметр (alpha)

3. КЛЮЧЕВЫЕ ПАРАМЕТРЫ:
   - Размер окна: 3-7 кадров оптимально
   - Alpha: 0.2-0.4 для стабильности

4. КОГДА СГЛАЖИВАНИЕ ПОМОГАЕТ:
   ✓ Медленно движущиеся объекты
   ✓ Стабильное освещение
   ✓ Высокая частота кадров (30+ FPS)

5. КОГДА УХУДШАЕТ:
   ✗ Быстрые движения
   ✗ Окклюзии
   ✗ Низкая частота кадров

6. РЕКОМЕНДАЦИИ:
   - Использовать exponential smoothing как baseline
   - Адаптировать alpha по скорости движения
   - Комбинировать с детектором окклюзий
''')


## 12. Сводка результатов


In [ ]:
# Создаем итоговую сводку
summary = {
    'video_info': {
        'path': str(video_path),
        'frames': len(frames),
        'fps': float(fps),
        'resolution': f"{frames[0].shape[1]}x{frames[0].shape[0]}"
    },
    'model': 'DeepLabV3-ResNet50',
    'target_class': 'person',
    'methods': [
        'Original (No Smoothing)',
        'Temporal Averaging (window=5)',
        'Median Filtering (window=5)',
        'Exponential Smoothing (alpha=0.3)'
    ],
    'metrics': all_results,
    'best_method': 'Exponential Smoothing',
    'improvement': {
        'iou_increase': f"{(results_exponential['mean_iou'] - results_original['mean_iou']):.4f}",
        'variance_reduction': f"{(results_original['mean_variance'] - results_exponential['mean_variance']):.6f}",
        'flickering_reduction': f"{(results_original['mean_flickering'] - results_exponential['mean_flickering']):.6f}"
    }
}

with open('results/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\\nВСЕ РЕЗУЛЬТАТЫ СОХРАНЕНЫ:')
print('  - results/first_frame.png')
print('  - results/original_masks.png')
print('  - results/metrics_comparison.png')
print('  - results/comparison_best.png')
print('  - results/metrics.json')
print('  - results/summary.json')
print('\\nДомашнее задание выполнено!')
